# Cómo leer un mapa de calor sin engañarte

**Newsletter**: Fútbol y Código  
**Semana**: 01 | **Tipo**: 🔧 Fundamento  
**Nivel**: ⭐⭐⭐☆☆

**Autor**: Juanje  
**Fecha**: 2026-03-18  
**Versión**: 1.0

## Descripción
Construimos mapas de calor de densidad (KDE) sobre los datos de Pedri en la Euro 2020. Aprendemos a generar heatmaps generales, filtrados por tipo de acción y comparativos entre partidos y jugadores, y sobre todo a interpretar qué revela y qué oculta esta técnica.

## Requisitos
- Python 3.10+
- Dependencias en `requirements.txt`
- Datos: StatsBomb Open Data (Euro 2020, `competition_id=55`, `season_id=43`)

## Tiempo estimado
- Lectura: 15 min  
- Ejecución: 5 min (primera vez ~3 min descargando datos)

---

## El Problema Real

Ves heatmaps todo el tiempo: en Twitter, en Wyscout, en informes de scouting que llegan a tu mesa. Son la visualización más reconocible del análisis de fútbol. Pero, ¿sabes qué dicen realmente?

Un heatmap de Pedri en la Euro 2020 muestra **dónde** tocó el balón. No muestra **cómo** lo jugó, ni si fue decisivo, ni si el espacio que ocupó fue por diseño táctico o circunstancia. Dos partidos con heatmaps idénticos pueden esconder rendimientos completamente opuestos.

Si no entiendes qué dice y qué **no** dice un heatmap, estás leyendo un mapa que puede engañarte. En este notebook vamos a construir heatmaps de forma rigurosa, compararlos entre partidos y jugadores, y —sobre todo— aprender a interpretarlos con criterio profesional.

In [ ]:
# =============================================================================
# REPRODUCIBILIDAD
# =============================================================================
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Competencia Objetivo

Después de este notebook, serás capaz de **generar heatmaps de densidad (KDE)**, interpretar qué revelan y qué ocultan, construir **comparativas entre partidos y jugadores**, y distinguir cuándo un heatmap informa una decisión de scouting vs cuándo te está engañando.

## Concepto / Fundamento

### ¿Qué es un heatmap de densidad?

Un heatmap transforma una lista de coordenadas (x, y) —cada acción de un jugador en el campo— en una **superficie continua de densidad**. La técnica detrás se llama **Kernel Density Estimation (KDE)**, y funciona así:

1. **Cada acción es una semilla.** Cada pase, recepción o recuperación tiene coordenadas en el campo (sistema StatsBomb: 120×80 metros, origen en la esquina inferior izquierda del campo atacante).
2. **El kernel suaviza.** Alrededor de cada punto, una función gaussiana extiende la "influencia" de esa acción a las zonas cercanas. Esto elimina el ruido de coordenadas exactas y revela patrones espaciales.
3. **La densidad se acumula.** Donde muchas acciones se superponen, los kernels se suman y la densidad crece. Esas son las zonas calientes del heatmap.

### ¿Por qué KDE y no un simple conteo por zonas?

La alternativa más básica sería dividir el campo en una cuadrícula y contar acciones por celda (hexbins o bins rectangulares). El problema: el resultado depende de dónde coloques la cuadrícula. Mueve los bordes unos metros y las zonas calientes cambian. KDE no tiene este problema porque no impone una cuadrícula fija; genera una superficie suave independiente de bordes arbitrarios.

### Interpretación futbolística

- **Zonas de alta densidad** indican áreas habituales de operación — no necesariamente las zonas más peligrosas.
- **Comparar heatmaps entre partidos** revela adaptación táctica: ¿el jugador cambió su posicionamiento según el rival?
- **Comparar entre jugadores** revela diferencias de rol: un mediocampista organizador y un interior de llegada operan en zonas distintas, y el heatmap lo hace visible de un vistazo.

### Lo que KDE **no** captura

Un KDE trata todas las acciones por igual. Un pase lateral de 5 metros y un pase de ruptura entre líneas pesan lo mismo en la superficie de densidad. Por eso, **filtrar por tipo de acción** antes de generar el heatmap es fundamental para obtener información táctica real. Lo veremos en la implementación.

## Implementación

### 3.0 Setup

In [ ]:
# =============================================================================
# IMPORTS Y CONFIGURACIÓN
# =============================================================================
from futbolycodigo.data_loaders import get_competition_matches, get_match_events
from futbolycodigo.viz_utils import (
    create_pitch, add_header, add_footer, plot_heatmap, create_comparison,
)
from futbolycodigo.branding import apply_style, get_theme

import pandas as pd
import matplotlib.pyplot as plt

# Aplica estilo FyC (tema light con césped, fuente Inter)
apply_style()

# Euro 2020 identifiers in StatsBomb Open Data
EURO_COMP_ID = 55
EURO_SEASON_ID = 43

### 3.1 Carga de datos

Cargamos los partidos de la Euro 2020 y filtramos los de España. Luego consolidamos todos los eventos de esos partidos para trabajar con el torneo completo.

In [ ]:
matches = get_competition_matches(EURO_COMP_ID, EURO_SEASON_ID)

# Spain's matches in the tournament
spain_matches = matches[
    (matches["home_team"] == "Spain") | (matches["away_team"] == "Spain")
].sort_values("match_date")

spain_matches[["match_id", "match_date", "home_team", "away_team"]]

In [ ]:
# Load events for all Spain matches (cached as parquet after first run)
spain_match_ids = spain_matches["match_id"].tolist()
all_events = pd.concat(
    [get_match_events(mid) for mid in spain_match_ids],
    ignore_index=True,
)

# Verify Pedri's exact name in the dataset
pedri_candidates = all_events[
    all_events["player"].str.contains("Pedri|Pedro González", case=False, na=False)
]["player"].unique()
print("Pedri name variants found:", pedri_candidates)

PLAYER_NAME = pedri_candidates[0]

In [ ]:
def extract_coordinates(events_df: pd.DataFrame) -> pd.DataFrame:
    """Extract x, y from the location column, dropping rows without location."""
    df = events_df.dropna(subset=["location"]).copy()
    df["x"] = df["location"].apply(lambda loc: loc[0])
    df["y"] = df["location"].apply(lambda loc: loc[1])
    return df

In [ ]:
# Filter Pedri's actions across the entire tournament
pedri_events = all_events[all_events["player"] == PLAYER_NAME].copy()
pedri_events = extract_coordinates(pedri_events)

print(f"Pedri — acciones con ubicación: {len(pedri_events)}")
print(f"\nDistribución por tipo de acción:")
print(pedri_events["type"].value_counts().head(10))

### 3.2 Heatmap general: todas las acciones de Pedri

Empezamos con la vista más amplia posible: **todas** las acciones de Pedri en el torneo, sin filtrar. Esto muestra su zona de influencia general, pero —como veremos después— mezcla contextos muy distintos.

In [ ]:
# --- Viz 1: Full-tournament heatmap (all actions) ---
fig = plot_heatmap(
    pedri_events["x"],
    pedri_events["y"],
    title="Pedri — Todas las acciones",
    subtitle="UEFA Euro 2020 | Torneo completo",
    orientation="vertical",
)
plt.show()

### 3.3 Heatmaps por tipo de acción

El heatmap general mezcla pases, recepciones, recuperaciones y conducciones en una sola superficie. Eso oculta información clave. Separar por tipo de acción revela **dónde distribuye** (pases), **dónde recibe** (recepciones), **dónde recupera** (recuperaciones) y **dónde conduce** (conducciones). Cuatro mapas, cuatro historias distintas del mismo jugador.

In [ ]:
# --- Viz 2: Heatmaps by action type (2x2 grid) ---
action_types = {
    "Pass": "Pases",
    "Ball Receipt*": "Recepciones",
    "Ball Recovery": "Recuperaciones",
    "Carry": "Conducciones",
}

theme = get_theme()
fig, axes, pitch = create_comparison(nrows=2, ncols=2, figsize=(12, 16))

for ax, (action_en, action_es) in zip(axes.flat, action_types.items()):
    subset = pedri_events[pedri_events["type"] == action_en]
    if len(subset) > 0:
        pitch.kdeplot(
            subset["x"], subset["y"], ax=ax,
            cmap="fyc_heat", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
        )
    ax.set_title(
        f"{action_es} (n={len(subset)})",
        fontsize=13, fontweight="bold", color=theme.accent,
    )

add_header(fig, "Pedri — Acciones por tipo", "UEFA Euro 2020 | Torneo completo")
add_footer(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.show()

### 3.4 Comparativa entre partidos: Pedri vs Croacia vs Pedri vs Italia

Un heatmap de torneo completo promedia todo. Pero un scout necesita saber: **¿Pedri cambia su posicionamiento según el rival?** Comparamos dos partidos con contextos tácticos opuestos:

- **España vs Croacia** (octavos de final): partido abierto, 5-3 tras prórroga, espacios amplios.
- **España vs Italia** (semifinal): partido cerrado, 1-1, definido en penaltis, pressing alto de Italia.

La hipótesis: el heatmap de Pedri debería ser más amplio y adelantado contra Croacia, más compacto y profundo contra Italia.

In [ ]:
# Identify match IDs dynamically from the matches DataFrame
croatia_match = spain_matches[
    spain_matches["home_team"].str.contains("Croatia")
    | spain_matches["away_team"].str.contains("Croatia")
]
italy_match = spain_matches[
    spain_matches["home_team"].str.contains("Italy")
    | spain_matches["away_team"].str.contains("Italy")
]

croatia_match_id = croatia_match["match_id"].iloc[0]
italy_match_id = italy_match["match_id"].iloc[0]

# Filter Pedri events per match
pedri_vs_croatia = pedri_events[pedri_events["match_id"] == croatia_match_id]
pedri_vs_italy = pedri_events[pedri_events["match_id"] == italy_match_id]

print(f"España vs Croacia (match_id={croatia_match_id}): {len(pedri_vs_croatia)} acciones")
print(f"España vs Italia  (match_id={italy_match_id}):  {len(pedri_vs_italy)} acciones")

In [ ]:
# --- Viz 3: Side-by-side match comparison ---
theme = get_theme()
fig, axes, pitch = create_comparison(ncols=2, figsize=(12, 8))

# Left: vs Croatia
pitch.kdeplot(
    pedri_vs_croatia["x"], pedri_vs_croatia["y"], ax=axes[0],
    cmap="fyc_heat", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
)
axes[0].set_title(
    f"vs Croacia — Octavos (n={len(pedri_vs_croatia)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

# Right: vs Italy
pitch.kdeplot(
    pedri_vs_italy["x"], pedri_vs_italy["y"], ax=axes[1],
    cmap="fyc_heat", fill=True, levels=100, thresh=0.25, zorder=3, alpha=0.6,
)
axes[1].set_title(
    f"vs Italia — Semifinal (n={len(pedri_vs_italy)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

add_header(fig, "Pedri — Comparativa entre partidos", "UEFA Euro 2020 | Octavos vs Semifinal")
add_footer(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.show()

### 3.5 Comparativa con otro mediocampista: Pedri vs Jorginho

Para que un heatmap tenga valor en scouting, necesita contexto. Comparar a Pedri con **Jorginho** (Italia) revela cómo dos mediocampistas de élite del mismo torneo ocupan el espacio de forma radicalmente distinta:

- **Pedri**: interior que opera en medios espacios, progresa con conducción y asociación en el último tercio.
- **Jorginho**: pivote profundo que orquesta desde delante de la defensa, rara vez aparece en zonas avanzadas.

Si ambos tuvieran el mismo heatmap, algo estaría mal en nuestro análisis. La diferencia es lo que valida la herramienta.

In [ ]:
# Load Italy's matches and Jorginho's events
italy_matches = matches[
    (matches["home_team"] == "Italy") | (matches["away_team"] == "Italy")
]
italy_match_ids = italy_matches["match_id"].tolist()

italy_events = pd.concat(
    [get_match_events(mid) for mid in italy_match_ids],
    ignore_index=True,
)

# Verify Jorginho's exact name
jorginho_candidates = italy_events[
    italy_events["player"].str.contains("Jorginho|Jorge Luiz|Frello", case=False, na=False)
]["player"].unique()
print("Jorginho name variants found:", jorginho_candidates)

COMPARISON_PLAYER = jorginho_candidates[0]

jorginho_events = italy_events[italy_events["player"] == COMPARISON_PLAYER].copy()
jorginho_events = extract_coordinates(jorginho_events)

print(f"\nJorginho — acciones con ubicación: {len(jorginho_events)}")

In [ ]:
# --- Viz 4: Pedri vs Jorginho — all actions ---
theme = get_theme()
fig, axes, pitch = create_comparison(ncols=2, figsize=(12, 8))

# Left: Pedri (cmap cálido)
pitch.kdeplot(
    pedri_events["x"], pedri_events["y"], ax=axes[0],
    cmap="fyc_heat", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
)
axes[0].set_title(
    f"Pedri — España (n={len(pedri_events)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

# Right: Jorginho (cmap azul para contraste visual)
pitch.kdeplot(
    jorginho_events["x"], jorginho_events["y"], ax=axes[1],
    cmap="Blues", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
)
axes[1].set_title(
    f"Jorginho — Italia (n={len(jorginho_events)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

add_header(fig, "Mediocampistas: perfiles espaciales distintos", "UEFA Euro 2020 | Torneo completo")
add_footer(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.show()

### 3.6 Zona de distribución: solo pases

Para afinar la comparativa, aislamos **solo los pases**. Esto elimina el ruido de acciones defensivas o recepciones y revela exclusivamente **desde dónde distribuye** cada jugador. Es la vista más útil para evaluar el rol táctico en la circulación del equipo.

In [ ]:
# --- Viz 5: Pedri vs Jorginho — passes only ---
pedri_passes = pedri_events[pedri_events["type"] == "Pass"]
jorginho_passes = jorginho_events[jorginho_events["type"] == "Pass"]

theme = get_theme()
fig, axes, pitch = create_comparison(ncols=2, figsize=(12, 8))

pitch.kdeplot(
    pedri_passes["x"], pedri_passes["y"], ax=axes[0],
    cmap="fyc_heat", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
)
axes[0].set_title(
    f"Pedri — Pases (n={len(pedri_passes)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

pitch.kdeplot(
    jorginho_passes["x"], jorginho_passes["y"], ax=axes[1],
    cmap="Blues", fill=True, levels=100, thresh=0.05, zorder=3, alpha=0.7,
)
axes[1].set_title(
    f"Jorginho — Pases (n={len(jorginho_passes)})",
    fontsize=13, fontweight="bold", color=theme.accent,
)

add_header(fig, "Zona de distribución: Pedri vs Jorginho", "Solo pases | UEFA Euro 2020")
add_footer(fig)
plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.show()

## Limitaciones y Judgment Call

### Lo que un heatmap no te dice

Un heatmap es una herramienta de **primer filtro**, no de veredicto final. Antes de tomar decisiones basadas en heatmaps, un analista profesional debe tener claras estas limitaciones:

**Sensibilidad al bandwidth.** El parámetro de suavizado (bandwidth) del KDE controla cuánto se expande cada punto. Un bandwidth alto produce un mapa difuso donde todo parece homogéneo; uno bajo produce picos aislados que exageran clusters puntuales. El valor por defecto de mplsoccer es razonable para la mayoría de análisis, pero en producción se ajusta según la cantidad de datos disponibles.

**Todas las acciones pesan igual.** Un pase lateral de 5 metros y un pase de ruptura entre líneas cuentan exactamente lo mismo en la superficie de densidad. Filtrar por tipo de acción mitiga esto parcialmente, pero dentro de cada categoría, la calidad de la acción es invisible para el KDE.

**Tamaño de muestra.** Un partido individual genera 50–100 acciones localizadas por jugador. Esa es una muestra pequeña para KDE: la densidad resultante puede ser ruidosa y poco representativa. El heatmap de torneo completo es más robusto, pero pierde la especificidad partido a partido. No hay solución perfecta; hay trade-offs que el analista debe gestionar.

**Posesión como variable confusora.** España tuvo más posesión que la mayoría de rivales en la Euro 2020. Eso significa que Pedri acumula más acciones que un jugador de un equipo defensivo, no porque sea "más activo" sino porque su equipo tiene más el balón. Comparar heatmaps crudos entre jugadores de equipos con posesiones muy distintas puede ser engañoso. Una normalización por minutos de posesión daría una imagen más justa.

**¿Cuándo usar un heatmap vs un scatter plot?** Para muestras grandes (torneo, temporada), el KDE es ideal. Para muestras pequeñas (un partido, una fase del juego), un scatter plot con transparencia puede ser más honesto porque muestra los puntos reales sin suavizado artificial. Clubes como Liverpool usan heatmaps como primera capa visual en sus informes de scouting, pero nunca como evidencia definitiva: siempre se complementan con métricas de impacto (xT, VAEP) y revisión de vídeo.

## Validación Académica

La estimación de densidad por kernel (KDE) es una técnica estadística consolidada desde los trabajos fundacionales de **Silverman (1986)**, cuyo libro *Density Estimation for Statistics and Data Analysis* estableció las bases teóricas para la selección automática de bandwidth y la elección de funciones kernel. La implementación que usamos en este notebook (vía scipy y mplsoccer) aplica un kernel gaussiano con bandwidth estimado por el método de Scott, que es óptimo para distribuciones unimodales y ofrece un buen compromiso entre sesgo y varianza para los volúmenes de datos típicos en football analytics (cientos a miles de acciones por jugador y temporada).

En el contexto específico del fútbol, **Fernández y Bornn (2018)** llevaron el concepto de densidad espacial varios pasos más allá con su modelo de *pitch control*: superficies continuas que estiman qué equipo controla cada zona del campo en cada instante. El heatmap de acciones que construimos aquí es un precursor más simple de esa idea — muestra dónde operó un jugador, no quién controla el espacio. Sin embargo, la intuición fundamental es la misma: transformar coordenadas discretas en superficies continuas que revelen patrones invisibles en los datos crudos.

Desde el lado aplicado, **StatsBomb** ha publicado extensamente sobre el uso de KDE en sus informes de scouting y análisis táctico, particularmente en su serie de investigación pública sobre análisis posicional. Su enfoque enfatiza que los heatmaps son herramientas de comunicación visual que funcionan mejor cuando se combinan con métricas de impacto (como Expected Threat o VAEP) y contexto táctico cualitativo. La academia respalda la técnica; el criterio profesional decide cuándo y cómo aplicarla.

In [ ]:
# =============================================================================
# INFORMACIÓN DEL ENTORNO
# =============================================================================
import importlib.metadata
import sys

print(f"Python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "matplotlib", "mplsoccer", "statsbombpy"]:
    try:
        print(f"{pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}: not installed")

## Siguiente Paso

El heatmap muestra **dónde**. Pero un scout necesita saber también **con quién**. ¿A quién busca Pedri cuando tiene el balón? ¿Quién le busca a él? ¿Es el centro de la circulación de España o un eslabón más en la cadena?

En el siguiente notebook construiremos la **red de pases de España** en la Euro 2020. Las redes de pases transforman los eventos en un grafo donde los nodos son jugadores y las aristas son pases entre ellos. Esto revela el rol real de Pedri en la estructura colectiva del equipo — algo que ningún heatmap individual puede capturar. Pasamos del análisis espacial individual al análisis relacional del equipo.